# Unstructured Scraper QA

Generic QA and first-pass EDA for scraper outputs under `data/unstructured/<source>/`. Set `SOURCE` in the setup cell to inspect `gefluegelnews`, `padi_web`, or another source that follows the backend scraper output contract.


## Setup


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


plt.style.use("ggplot")
plt.rcParams.update({"figure.figsize": [14, 7], "text.color": (0.25, 0.25, 0.25)})
pd.options.display.max_columns = 160
pd.options.display.max_rows = 200
pd.options.display.max_colwidth = 140

SOURCE = "gefluegelnews"
EXPECTED_JSONL = {
    "manifest": "manifest.jsonl",
    "articles": "articles.jsonl",
    "parse_errors": "parse_errors.jsonl",
    "disease_articles": "disease_articles.jsonl",
    "disease_reports": "disease_reports.jsonl",
    "disease_reports_embeddings": "disease_reports_embeddings.jsonl",
}


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError(
        "Could not locate repository root with pyproject.toml and data/."
    )


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "data" / "unstructured"
SOURCE_DIR = DATA_ROOT / SOURCE

if not SOURCE_DIR.exists():
    available = sorted(path.name for path in DATA_ROOT.iterdir() if path.is_dir())
    raise FileNotFoundError(
        f"No scraper output directory found for SOURCE={SOURCE!r}. Available sources: {available}"
    )

SOURCE_DIR

PosixPath('/Volumes/1TB Home SSD/GitHub/_ GitHub generell/GovTech-Tierseuchen/data/unstructured/gefluegelnews')

In [ ]:
def read_jsonl(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict] = []
    errors: list[dict] = []
    if not path.exists():
        return pd.DataFrame(), pd.DataFrame()

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError as exc:
                errors.append(
                    {"line_number": line_number, "error": str(exc), "text": text[:300]}
                )
                continue
            if isinstance(parsed, dict):
                rows.append(parsed)
            else:
                rows.append({"value": parsed})

    return pd.json_normalize(rows), pd.DataFrame(errors)


def stable_value(value: object) -> object:
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return value


def nunique_nested(series: pd.Series) -> int:
    return series.dropna().map(stable_value).nunique(dropna=True)


def schema_summary(data: pd.DataFrame) -> pd.DataFrame:
    if data.empty:
        return pd.DataFrame(columns=["dtype", "missing", "missing_pct", "n_unique"])
    return pd.DataFrame(
        {
            "dtype": data.dtypes.astype(str),
            "missing": data.isna().sum(),
            "missing_pct": data.isna().mean().mul(100).round(1),
            "n_unique": data.apply(nunique_nested),
        }
    ).sort_values(["missing_pct", "n_unique"], ascending=[False, False])


def top_counts(data: pd.DataFrame, column: str, n: int = 15) -> pd.DataFrame:
    if data.empty or column not in data.columns:
        return pd.DataFrame(columns=[column, "records"])
    return (
        data[column]
        .fillna("Missing")
        .value_counts(dropna=False)
        .head(n)
        .rename_axis(column)
        .reset_index(name="records")
    )


def parse_datetime_columns(data: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    parsed = data.copy()
    for column in columns:
        if column in parsed.columns:
            parsed[column] = pd.to_datetime(parsed[column], errors="coerce", utc=True)
    return parsed


def list_length(value: object) -> int:
    return len(value) if isinstance(value, list) else 0


def flatten_list_values(series: pd.Series) -> list[object]:
    values: list[object] = []
    for item in series.dropna():
        if isinstance(item, list):
            values.extend(item)
        else:
            values.append(item)
    return values


def plot_top_counts(
    data: pd.DataFrame, specs: list[tuple[str, int, str]], cols: int = 2
) -> None:
    available_specs = [spec for spec in specs if spec[0] in data.columns]
    if not available_specs:
        print("No requested columns are available for plotting.")
        return

    rows = -(-len(available_specs) // cols)
    fig, axes = plt.subplots(
        rows, cols, figsize=(7.5 * cols, 4.8 * rows), squeeze=False
    )
    for axis, (column, n, title) in zip(axes.ravel(), available_specs):
        counts = top_counts(data, column, n=n)
        sns.barplot(data=counts, y=column, x="records", ax=axis, color="#4C78A8")
        axis.set_title(title)
        axis.set_xlabel("Records")
        axis.set_ylabel("")
    for axis in axes.ravel()[len(available_specs) :]:
        axis.set_visible(False)
    plt.tight_layout()


def monthly_counts(
    data: pd.DataFrame, date_column: str, label: str = "records"
) -> pd.DataFrame:
    if data.empty or date_column not in data.columns:
        return pd.DataFrame(columns=["month", label])
    return (
        data.dropna(subset=[date_column])
        .assign(
            month=lambda frame: frame[date_column].dt.to_period("M").dt.to_timestamp()
        )
        .groupby("month")
        .size()
        .rename(label)
        .reset_index()
    )

## File Coverage


In [ ]:
artifact_rows = []
for name, filename in EXPECTED_JSONL.items():
    path = SOURCE_DIR / filename
    artifact_rows.append(
        {
            "artifact": name,
            "path": path.relative_to(REPO_ROOT),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / 1_000_000, 2)
            if path.exists()
            else None,
            "modified_at": pd.to_datetime(path.stat().st_mtime, unit="s")
            if path.exists()
            else pd.NaT,
        }
    )

artifact_status = pd.DataFrame(artifact_rows)
display(artifact_status)

In [ ]:
frames: dict[str, pd.DataFrame] = {}
jsonl_errors: dict[str, pd.DataFrame] = {}

for name, filename in EXPECTED_JSONL.items():
    frame, errors = read_jsonl(SOURCE_DIR / filename)
    frames[name] = frame
    jsonl_errors[name] = errors

manifest = frames["manifest"]
articles = parse_datetime_columns(
    frames["articles"], ["publication_date", "retrieved_at"]
)
parse_errors = frames["parse_errors"]
disease_articles = frames["disease_articles"]
disease_reports = parse_datetime_columns(
    frames["disease_reports"],
    [
        "source_publication_date",
        "source_retrieved_at",
        "confirmation_date",
        "result_date",
    ],
)
disease_reports_embeddings = parse_datetime_columns(
    frames["disease_reports_embeddings"],
    [
        "source_publication_date",
        "source_retrieved_at",
        "confirmation_date",
        "result_date",
    ],
)

row_counts = pd.DataFrame(
    {
        "artifact": list(frames),
        "rows": [len(frame) for frame in frames.values()],
        "columns": [len(frame.columns) for frame in frames.values()],
        "json_decode_errors": [len(jsonl_errors[name]) for name in frames],
    }
)
display(row_counts)

In [ ]:
schema_tables = []
for name, frame in frames.items():
    if frame.empty:
        continue
    summary = schema_summary(frame).reset_index(names="column")
    summary.insert(0, "artifact", name)
    schema_tables.append(summary)

schema = (
    pd.concat(schema_tables, ignore_index=True) if schema_tables else pd.DataFrame()
)
display(schema.head(80))

## Pipeline Funnel


In [ ]:
funnel = row_counts[
    row_counts["artifact"].isin(
        [
            "manifest",
            "articles",
            "parse_errors",
            "disease_articles",
            "disease_reports",
            "disease_reports_embeddings",
        ]
    )
].copy()
display(funnel)

axis = sns.barplot(data=funnel, y="artifact", x="rows", color="#4C78A8")
axis.set_title(f"{SOURCE}: scraper pipeline row counts")
axis.set_xlabel("Rows")
axis.set_ylabel("")
plt.tight_layout()

In [ ]:
key_checks = []
for name, frame in frames.items():
    for column in [
        "source_link",
        "canonical_url",
        "content_hash",
        "report_id",
        "source_document_id",
    ]:
        if column in frame.columns:
            key_checks.append(
                {
                    "artifact": name,
                    "column": column,
                    "non_null": frame[column].notna().sum(),
                    "unique": frame[column].nunique(dropna=True),
                    "duplicates": frame[column].dropna().duplicated().sum(),
                }
            )

display(pd.DataFrame(key_checks))

In [ ]:
decode_error_tables = []
for name, errors in jsonl_errors.items():
    if errors.empty:
        continue
    with_name = errors.copy()
    with_name.insert(0, "artifact", name)
    decode_error_tables.append(with_name)

decode_errors = (
    pd.concat(decode_error_tables, ignore_index=True)
    if decode_error_tables
    else pd.DataFrame()
)
display(decode_errors.head(20))

## Article EDA


In [ ]:
if articles.empty:
    print("No articles.jsonl rows loaded.")
else:
    article_dates = pd.DataFrame(
        {
            "date_column": [
                column
                for column in ["publication_date", "retrieved_at"]
                if column in articles.columns
            ],
            "min_date": [
                articles[column].min()
                for column in ["publication_date", "retrieved_at"]
                if column in articles.columns
            ],
            "max_date": [
                articles[column].max()
                for column in ["publication_date", "retrieved_at"]
                if column in articles.columns
            ],
            "missing": [
                articles[column].isna().sum()
                for column in ["publication_date", "retrieved_at"]
                if column in articles.columns
            ],
        }
    )
    display(article_dates)
    display(schema_summary(articles).head(40))

In [ ]:
if not articles.empty:
    plot_top_counts(
        articles,
        [
            ("category", 15, "Top article categories"),
            ("author", 15, "Top article authors"),
            ("source_name", 10, "Source names"),
            ("source_attribution", 15, "Source attributions"),
        ],
    )

In [ ]:
if not articles.empty and "publication_date" in articles.columns:
    article_months = monthly_counts(articles, "publication_date", label="articles")
    display(article_months.tail(24))
    axis = sns.lineplot(data=article_months, x="month", y="articles", marker="o")
    axis.set_title(f"{SOURCE}: articles by publication month")
    axis.set_xlabel("Publication month")
    axis.set_ylabel("Articles")
    plt.xticks(rotation=45)
    plt.tight_layout()

In [ ]:
if not articles.empty and "fulltext" in articles.columns:
    article_text = articles.assign(
        fulltext_chars=articles["fulltext"].fillna("").str.len(),
        fulltext_words=articles["fulltext"].fillna("").str.split().str.len(),
    )
    display(
        article_text[["title", "publication_date", "fulltext_chars", "fulltext_words"]]
        .sort_values("fulltext_words")
        .head(10)
    )
    display(
        article_text[["title", "publication_date", "fulltext_chars", "fulltext_words"]]
        .sort_values("fulltext_words", ascending=False)
        .head(10)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(data=article_text, x="fulltext_words", bins=40, ax=axes[0])
    axes[0].set_title("Article word counts")
    sns.boxplot(data=article_text, x="fulltext_words", ax=axes[1], color="#4C78A8")
    axes[1].set_title("Article word count spread")
    plt.tight_layout()

In [ ]:
if not articles.empty and "keywords" in articles.columns:
    keyword_counts = Counter(
        str(value).strip()
        for value in flatten_list_values(articles["keywords"])
        if str(value).strip()
    )
    top_keywords = pd.DataFrame(
        keyword_counts.most_common(40), columns=["keyword", "articles"]
    )
    display(top_keywords.head(25))
    if not top_keywords.empty:
        axis = sns.barplot(
            data=top_keywords.head(20), y="keyword", x="articles", color="#4C78A8"
        )
        axis.set_title("Top article keywords")
        axis.set_xlabel("Articles")
        axis.set_ylabel("")
        plt.tight_layout()

## Disease Filter QA


In [ ]:
if disease_articles.empty:
    print("No disease_articles.jsonl rows loaded.")
else:
    display(schema_summary(disease_articles).head(50))
    plot_top_counts(
        disease_articles,
        [
            ("article.category", 15, "Relevant article categories"),
            ("relevance.score", 12, "Relevance scores"),
            ("relevance.filter_version", 10, "Filter versions"),
        ],
    )

In [ ]:
if not disease_articles.empty and "relevance.matched_terms" in disease_articles.columns:
    matched_term_counts = Counter(
        str(value).strip()
        for value in flatten_list_values(disease_articles["relevance.matched_terms"])
        if str(value).strip()
    )
    terms = pd.DataFrame(
        matched_term_counts.most_common(), columns=["matched_term", "articles"]
    )
    display(terms.head(40))
    if not terms.empty:
        axis = sns.barplot(
            data=terms.head(25), y="matched_term", x="articles", color="#4C78A8"
        )
        axis.set_title("Disease filter matched terms")
        axis.set_xlabel("Articles")
        axis.set_ylabel("")
        plt.tight_layout()

In [ ]:
if not parse_errors.empty:
    display(parse_errors.head(20))
else:
    print("No parser failures recorded in parse_errors.jsonl.")

## Disease Report QA


In [ ]:
if disease_reports.empty:
    print("No disease_reports.jsonl rows loaded.")
else:
    display(schema_summary(disease_reports).head(80))
    plot_top_counts(
        disease_reports,
        [
            ("disease_name", 15, "Disease names"),
            ("country_or_territory", 15, "Countries or territories"),
            ("extraction_confidence", 10, "Extraction confidence"),
            ("relevance_level", 10, "Relevance levels"),
            ("extraction_status", 10, "Extraction status"),
            ("extraction_method", 10, "Extraction methods"),
        ],
    )

In [ ]:
important_report_fields = [
    "report_id",
    "source_link",
    "source_publication_date",
    "disease_name",
    "country_or_territory",
    "situation_month",
    "situation_key",
    "extraction_confidence",
    "raw_relevance_evidence",
    "evidence_snippets",
]

if not disease_reports.empty:
    available_fields = [
        field for field in important_report_fields if field in disease_reports.columns
    ]
    missingness = (
        disease_reports[available_fields]
        .isna()
        .mean()
        .mul(100)
        .round(1)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "field"})
        .sort_values("missing_pct", ascending=False)
    )
    display(missingness)

    mostly_missing = missingness[missingness["missing_pct"] >= 80]
    display(mostly_missing)

In [ ]:
if not disease_reports.empty:
    list_metrics = []
    for column in [
        "evidence_snippets",
        "control_measures",
        "prevention_measures",
        "research_references",
    ]:
        if column in disease_reports.columns:
            lengths = disease_reports[column].map(list_length)
            list_metrics.append(
                {
                    "column": column,
                    "min_items": lengths.min(),
                    "median_items": lengths.median(),
                    "max_items": lengths.max(),
                    "zero_item_rows": int((lengths == 0).sum()),
                }
            )
    display(pd.DataFrame(list_metrics))

    if "situation_month" in disease_reports.columns:
        situation_months = top_counts(disease_reports, "situation_month", n=30)
        display(situation_months)
        axis = sns.barplot(
            data=situation_months.head(24),
            y="situation_month",
            x="records",
            color="#4C78A8",
        )
        axis.set_title("Reports by situation month")
        axis.set_xlabel("Reports")
        axis.set_ylabel("")
        plt.tight_layout()

In [ ]:
if not disease_reports.empty:
    numeric_columns = [
        column
        for column in [
            "cases",
            "dead",
            "killed",
            "slaughtered",
            "susceptible",
            "vaccinated",
            "latitude",
            "longitude",
        ]
        if column in disease_reports.columns
    ]
    if numeric_columns:
        display(disease_reports[numeric_columns].describe(include="all"))

    sample_columns = [
        column
        for column in [
            "report_id",
            "source_publication_date",
            "disease_name",
            "country_or_territory",
            "extraction_confidence",
            "relevance_level",
            "source_document_title",
            "raw_relevance_evidence",
        ]
        if column in disease_reports.columns
    ]
    display(disease_reports[sample_columns].head(25))

## Embedding and Enrichment QA


In [ ]:
if disease_reports_embeddings.empty:
    print("No disease_reports_embeddings.jsonl rows loaded.")
else:
    display(schema_summary(disease_reports_embeddings).head(80))

    comparison = {
        "disease_reports_rows": len(disease_reports),
        "embedding_rows": len(disease_reports_embeddings),
    }
    if (
        "report_id" in disease_reports.columns
        and "report_id" in disease_reports_embeddings.columns
    ):
        report_ids = set(disease_reports["report_id"].dropna())
        embedding_report_ids = set(disease_reports_embeddings["report_id"].dropna())
        comparison.update(
            {
                "reports_without_embedding_row": len(report_ids - embedding_report_ids),
                "embedding_rows_without_report": len(embedding_report_ids - report_ids),
                "duplicate_embedding_report_ids": int(
                    disease_reports_embeddings["report_id"].dropna().duplicated().sum()
                ),
            }
        )
    display(pd.DataFrame([comparison]).T.rename(columns={0: "value"}))

In [ ]:
embedding_candidate_columns = [
    column
    for column in disease_reports_embeddings.columns
    if column.lower() in {"embedding", "embeddings", "vector", "vectors"}
    or column.lower().endswith("embedding")
    or column.lower().endswith("embeddings")
]

if disease_reports_embeddings.empty:
    pass
elif not embedding_candidate_columns:
    print("No embedding vector columns detected. Available enrichment columns:")
    display(pd.DataFrame({"column": disease_reports_embeddings.columns}))
else:
    vector_metrics = []
    for column in embedding_candidate_columns:
        lengths = disease_reports_embeddings[column].map(list_length)
        vector_metrics.append(
            {
                "column": column,
                "non_null": disease_reports_embeddings[column].notna().sum(),
                "min_dimension": lengths[lengths > 0].min()
                if (lengths > 0).any()
                else None,
                "median_dimension": lengths[lengths > 0].median()
                if (lengths > 0).any()
                else None,
                "max_dimension": lengths.max(),
                "malformed_or_missing": int((lengths == 0).sum()),
            }
        )
    display(pd.DataFrame(vector_metrics))

## Records to Inspect


In [ ]:
if not disease_reports.empty:
    candidate_columns = [
        "report_id",
        "source_link",
        "source_publication_date",
        "disease_name",
        "country_or_territory",
        "situation_month",
        "extraction_confidence",
        "relevance_level",
        "source_document_title",
        "raw_relevance_evidence",
    ]
    columns = [
        column for column in candidate_columns if column in disease_reports.columns
    ]
    display(
        disease_reports[columns].sample(min(10, len(disease_reports)), random_state=42)
    )
elif not disease_articles.empty:
    candidate_columns = [
        "article.title",
        "article.publication_date",
        "article.category",
        "relevance.score",
        "relevance.matched_terms",
        "article.source_link",
    ]
    columns = [
        column for column in candidate_columns if column in disease_articles.columns
    ]
    display(
        disease_articles[columns].sample(
            min(10, len(disease_articles)), random_state=42
        )
    )
elif not articles.empty:
    candidate_columns = [
        "title",
        "publication_date",
        "category",
        "source_link",
        "description",
    ]
    columns = [column for column in candidate_columns if column in articles.columns]
    display(articles[columns].sample(min(10, len(articles)), random_state=42))
else:
    print("No loaded rows available for inspection.")